# Policy gradients (REINFORCE)

_Modern Deep Learning & AI_

**Skip the value table. Directly raise the chance of actions that led to a big reward.**

DQN learns values, then acts greedily. Policy gradients skip that. They learn the policy itself: the probability of each action.

     The rule is wonderfully direct: if an action led to a high reward, make it more likely next time. If it led to a low reward, make it less likely.

---

This notebook is a practice scaffold for the **Policy gradients (REINFORCE)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical

class Policy(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 32), nn.ReLU(),
            nn.Linear(32, n_actions))            # logits over actions
    def forward(self, s):
        return self.net(s)

policy = Policy(state_dim=4, n_actions=2)
opt = torch.optim.Adam(policy.parameters(), lr=1e-2)

# synthetic episode: 5 states visited, with a per-step return G
states  = torch.randn(5, 4)
returns = torch.tensor([2.0, 1.5, 1.0, -0.5, -1.0])
returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # baseline

logits = policy(states)
dist   = Categorical(logits=logits)
actions = dist.sample()                          # actions actually taken
log_probs = dist.log_prob(actions)               # log pi(a | s)

loss = -(log_probs * returns).sum()              # REINFORCE objective
opt.zero_grad(); loss.backward(); opt.step()
print(float(loss))

## Visualize the data & results

_On the same corridor task, does a REINFORCE policy learn? Does the return G grow as we train?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
N, GOAL = 8, 7                       # same corridor task, goal reward = +10
theta = np.zeros((N, 2))            # softmax policy logits per state
lr, gamma = 0.05, 0.95
returns_hist = []
for ep in range(300):
    s, traj, G = 0, [], 0.0
    for _ in range(30):
        p = np.exp(theta[s] - theta[s].max()); p /= p.sum()   # softmax pi(.|s)
        a = rng.choice(2, p=p)
        s2 = min(N - 1, s + 1) if a == 1 else max(0, s - 1)
        r = 10.0 if s2 == GOAL else -0.1
        traj.append((s, a, r)); G += r; s = s2
        if s2 == GOAL:
            break
    Gt, baseline = 0.0, G / len(traj)         # mean-reward baseline cuts variance
    for s, a, r in reversed(traj):            # walk trajectory backwards
        Gt = r + gamma * Gt
        p = np.exp(theta[s] - theta[s].max()); p /= p.sum()
        grad = -p; grad[a] += 1.0             # d log pi / d theta
        theta[s] += lr * (Gt - baseline) * grad
    returns_hist.append(G)

sm = np.convolve(returns_hist, np.ones(20) / 20, mode='valid')
plt.figure(figsize=(6, 4))
plt.plot(sm, color='#7ee787', label='return G')
plt.xlabel('episode'); plt.ylabel('smoothed return G')
plt.title('REINFORCE on 8-state corridor'); plt.legend()
plt.tight_layout(); plt.show()